# Training Pipeline - Part 3: BERT Transformer Training
## Cyber AI Agent v2.0.0

**Purpose:** Train transformer-based classifier using BERT (Layer 4)

**Technique:** Transfer learning with fine-tuning

**Inputs:**
- X_train_scaled.pkl (raw features for NLP text generation)
- X_val_scaled.pkl
- X_test_scaled.pkl
- y_train.pkl
- y_val.pkl
- y_test.pkl

**Outputs:**
- bert_classifier/ (fine-tuned model directory)
- bert_metrics.json

---

## 1. Setup and Data Loading

In [4]:
import os
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('USE_FLAX', '0')

import torch
import numpy as np
import pandas as pd
import joblib
import json
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from torch.utils.data import Dataset as TorchDataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")

# Load preprocessed data
scaler = joblib.load('../trained_models/scaler.pkl')

X_train = pd.DataFrame(
    scaler.inverse_transform(
        joblib.load('../datasets/X_train_scaled.pkl')
    ),
    columns=joblib.load('../datasets/X_train_scaled.pkl').columns
)

X_val = pd.DataFrame(
    scaler.inverse_transform(
        joblib.load('../datasets/X_val_scaled.pkl')
    ),
    columns=joblib.load('../datasets/X_val_scaled.pkl').columns
)

X_test = pd.DataFrame(
    scaler.inverse_transform(
        joblib.load('../datasets/X_test_scaled.pkl')
    ),
    columns=joblib.load('../datasets/X_test_scaled.pkl').columns
)
y_train = joblib.load('../datasets/y_train.pkl')
y_val = joblib.load('../datasets/y_val.pkl')
y_test = joblib.load('../datasets/y_test.pkl')

print(f"✅ Data loaded: {X_train.shape[0]} training, {X_val.shape[0]} validation, {X_test.shape[0]} test samples")

# Detect number of unique labels
num_classes = len(np.unique(y_train))
print(f"ℹ️  Detected {num_classes} unique labels in training data")
print(f"   Labels: {sorted(np.unique(y_train))}")


✅ Using device: cuda
✅ Data loaded: 1932175 training, 241522 validation, 241522 test samples
ℹ️  Detected 15 unique labels in training data
   Labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


In [5]:
import os
import json
from datetime import datetime

print("\n🔍 MODEL STATUS CHECK")
print("=" * 80)

model_path = '../trained_models/bert_classifier'
metrics_path = '../trained_models/bert_metrics.json'

model_exists = os.path.exists(model_path)
metrics_exists = os.path.exists(metrics_path)

print(f"\n1. Model artifacts:")
print(f"   bert_classifier/ exists: {model_exists}")
print(f"   bert_metrics.json exists: {metrics_exists}")

if model_exists:
    files = os.listdir(model_path)
    print(f"   Files in bert_classifier/: {files}")
    
if metrics_exists:
    metrics = json.load(open(metrics_path))
    print(f"\n2. Existing metrics (timestamp check):")
    mod_time = datetime.fromtimestamp(os.path.getmtime(metrics_path))
    print(f"   Last modified: {mod_time}")
    print(f"   Test F1: {metrics.get('test_f1_score', 'N/A')}")
    print(f"   Training time: {metrics.get('training_time', 'N/A'):.1f}s")
    print(f"   ⚠️  If old timestamp → model is from previous interrupted session")
else:
    print(f"\n2. No existing metrics - fresh training required")


🔍 MODEL STATUS CHECK

1. Model artifacts:
   bert_classifier/ exists: True
   bert_metrics.json exists: True
   Files in bert_classifier/: ['checkpoint-11436', 'checkpoint-22872', 'checkpoint-34308', 'config.json', 'model.safetensors', 'runs', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.txt']

2. Existing metrics (timestamp check):
   Last modified: 2026-05-28 06:30:17.040985
   Test F1: N/A
   Training time: 12968.8s
   ⚠️  If old timestamp → model is from previous interrupted session


In [6]:
print(X_train.head())
print(X_train.describe())

   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0           52647.0       999381.0                2.0                     0.0   
1              53.0          134.0                2.0                     2.0   
2              53.0        61599.0                1.0                     1.0   
3             443.0      4047702.0                5.0                     1.0   
4              53.0       155677.0                2.0                     2.0   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                         12.0                          0.0   
1                         70.0                        102.0   
2                         49.0                        129.0   
3                        135.0                         46.0   
4                         84.0                        318.0   

   Fwd Packet Length Mean  Bwd Packet Length Mean  Flow Bytes/s  \
0                     6.0                     0.0  1.200743e+01   


## 2. Generate NLP Text from Network Features

In [7]:
print("\n📝 GENERATING NLP DESCRIPTIONS")
print("=" * 80)


def generate_nlp_description(row):
    """Convert numeric network features to clean, BERT-friendly cybersecurity text."""

    # Safe extraction
    def safe(x):
        return float(x) if x is not None else 0.0

    duration = int(max(row.get('Flow Duration', 0), 1))
    fwd_pkts = int(row.get('Total Fwd Packets', 0))
    bwd_pkts = int(row.get('Total Backward Packets', 0))

    fwd_bytes = int(row.get('Total Length of Fwd Packets', 0))
    bwd_bytes = int(row.get('Total Length of Bwd Packets', 0))

    # ✅ FIX: Ensure non-negative flow rates (handles negative values in data)
    flow_bps = max(safe(row.get('Flow Bytes/s', 0)), 0.0)
    flow_pps = max(safe(row.get('Flow Packets/s', 0)), 0.0)

    avg_pkt = max(safe(row.get('Average Packet Size', 0)), 0.0)
    port = int(row.get('Destination Port', 0))

    # Derived features
    total_pkts = fwd_pkts + bwd_pkts
    pkt_ratio = fwd_pkts / max(bwd_pkts, 1)

    # Simple semantic labels
    if flow_pps > 1000:
        traffic_type = "high traffic burst pattern"
    elif flow_pps > 300:
        traffic_type = "moderate network activity"
    else:
        traffic_type = "normal network flow"

    payload_type = "large payload" if avg_pkt > 800 else "small payload"
    flow_balance = "skewed traffic flow" if pkt_ratio > 2 or pkt_ratio < 0.5 else "balanced traffic flow"

    text = (
        f"Network flow detected on destination port {port}. "
        f"Duration is {duration} ms with {total_pkts} total packets. "
        f"Forward packets: {fwd_pkts} ({fwd_bytes} bytes), "
        f"Backward packets: {bwd_pkts} ({bwd_bytes} bytes). "
        f"Traffic rate is {flow_bps:.2f} bytes per second and {flow_pps:.2f} packets per second. "
        f"Packet size indicates {payload_type}. "
        f"Flow behavior shows {flow_balance}. "
        f"Overall pattern indicates {traffic_type}."
    )

    return " ".join(text.split())


def generate_texts(frame):
    columns = list(frame.columns)
    return [generate_nlp_description(dict(zip(columns, values))) for values in frame.itertuples(index=False, name=None)]


# Generate datasets
train_texts = generate_texts(X_train)
val_texts = generate_texts(X_val)
test_texts = generate_texts(X_test)

print(f"\n1. Generated {len(train_texts)} training descriptions")
print(f"2. Generated {len(val_texts)} validation descriptions")
print(f"3. Generated {len(test_texts)} test descriptions")
print(f"\n4. Example training text:")
print(f"   {train_texts[0][:200]}...")


📝 GENERATING NLP DESCRIPTIONS

1. Generated 1932175 training descriptions
2. Generated 241522 validation descriptions
3. Generated 241522 test descriptions

4. Example training text:
   Network flow detected on destination port 52647. Duration is 999381 ms with 2 total packets. Forward packets: 2 (12 bytes), Backward packets: 0 (0 bytes). Traffic rate is 12.01 bytes per second and 2....


## 3. Prepare BERT Dataset

In [8]:
print("\n🔄 PREPARING BERT DATASET")
print("=" * 80)

# Load pre-trained BERT tokenizer
model_name = 'prajjwal1/bert-tiny'  # Lightweight BERT variant (4M params)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"\n1. Loaded tokenizer: {model_name}")
print(f"   Vocab size: {tokenizer.vocab_size}")

# Custom PyTorch Dataset class (avoids pyarrow DLL issue)
class TextDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = TextDataset(train_texts, y_train.values, tokenizer)
val_dataset = TextDataset(val_texts, y_val.values, tokenizer)
test_dataset = TextDataset(test_texts, y_test.values, tokenizer)

print(f"\n2. Created training dataset: {len(train_dataset)} samples")
print(f"   Created validation dataset: {len(val_dataset)} samples")
print(f"   Created test dataset: {len(test_dataset)} samples")

print(f"\n3. Tokenization configured:")
print(f"   Max sequence length: 128 tokens")
print(f"   ✅ Ready for training")


🔄 PREPARING BERT DATASET

1. Loaded tokenizer: prajjwal1/bert-tiny
   Vocab size: 30522

2. Created training dataset: 1932175 samples
   Created validation dataset: 241522 samples
   Created test dataset: 241522 samples

3. Tokenization configured:
   Max sequence length: 128 tokens
   ✅ Ready for training


## 4. Load Pre-trained BERT and Fine-tune

In [ ]:
print("\n🤖 FINE-TUNING BERT")
print("=" * 80)

from transformers import AutoModelForSequenceClassification

# Load pre-trained BERT model with correct number of labels
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes  # Use detected number of classes
)

model = model.to(device)

print(f"\n1. Loaded pre-trained BERT:")
print(f"   Model: {model_name}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Device: {device}")
print(f"   Number of classes: {num_classes}")

# ✅ OPTIMIZED TRAINING ARGUMENTS FOR FULL DATASET
training_args = TrainingArguments(
    output_dir='../trained_models/bert_classifier',
    learning_rate=2e-5,
    per_device_train_batch_size=16,        # Good balance for full dataset
    per_device_eval_batch_size=32,
    num_train_epochs=3,                    # Full 3 epochs for complete dataset
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy="steps",
    logging_steps=500,   # 🔥 show updates frequently
    disable_tqdm=False,  # 🔥 FORCE progress bar ON
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
                         # Adjusted for full dataset (1.9M samples)
    fp16=True,                             # Enable mixed precision (2× speedup)
    dataloader_num_workers=4,              # Parallel data loading
    gradient_accumulation_steps=2,         # Effective batch size = 32
    save_total_limit=2,                    # Keep only 2 checkpoints
    seed=42
)

print(f"\n2. Training configuration (FULL DATASET):")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Train batch size: 16 (effective: 32 with accumulation)")
print(f"   Eval batch size: 32")
print(f"   Epochs: 3 (full training)")
print(f"   Training samples: {len(y_train):,}")
print(f"   Validation samples: {len(y_val):,}")
print(f"   Mixed precision (fp16): ✓ ENABLED")
print(f"   Data workers: 4")
print(f"   Gradient accumulation: 2 steps")
print(f"   Expected duration: 3-5 hours on GPU with optimizations")

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer)
)

print(f"\n3. Starting fine-tuning on FULL DATASET...")
import time
start_time = time.time()

# Train
trainer.train()

training_time = time.time() - start_time
print(f"\n✅ Fine-tuning complete in {training_time:.2f} seconds ({training_time/60:.1f} minutes)")
print(f"   Training on {len(y_train):,} samples × 3 epochs")



🤖 FINE-TUNING BERT


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



1. Loaded pre-trained BERT:
   Model: prajjwal1/bert-tiny
   Parameters: 4,387,855
   Device: cuda
   Number of classes: 15

2. Training configuration (FULL DATASET):
   Learning rate: 2e-05
   Train batch size: 16 (effective: 32 with accumulation)
   Eval batch size: 32
   Epochs: 3 (full training)
   Training samples: 1,932,175
   Validation samples: 241,522
   Mixed precision (fp16): ✓ ENABLED
   Data workers: 4
   Gradient accumulation: 2 steps
   Expected duration: 3-5 hours on GPU with optimizations

3. Starting fine-tuning on FULL DATASET...


## 5. Evaluate BERT Model

In [ ]:
print("\n📊 BERT EVALUATION")
print("=" * 80)


def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }

print("\n1. Validation set used for model selection during training.")

# Validation metrics
val_predictions = trainer.predict(val_dataset)
val_pred = np.argmax(val_predictions.predictions, axis=1)
val_metrics = compute_metrics(y_val, val_pred)

print(f"\n2. Validation Performance:")
print(f"   Accuracy: {val_metrics['accuracy']:.4f}")
print(f"   Precision (weighted): {val_metrics['precision']:.4f}")
print(f"   Recall (weighted): {val_metrics['recall']:.4f}")
print(f"   F1-Score (weighted): {val_metrics['f1']:.4f}")

print(f"\n3. Validation Classification Report:")
print(classification_report(y_val, val_pred, zero_division=0))

# Test metrics
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
test_metrics = compute_metrics(y_test, y_pred)

print(f"\n4. Test Performance:")
print(f"   Accuracy: {test_metrics['accuracy']:.4f}")
print(f"   Precision (weighted): {test_metrics['precision']:.4f}")
print(f"   Recall (weighted): {test_metrics['recall']:.4f}")
print(f"   F1-Score (weighted): {test_metrics['f1']:.4f}")

print(f"\n5. Test Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


📊 BERT EVALUATION

1. Validation set used for model selection during training.


NameError: name 'trainer' is not defined

## 6. Save Model and Metrics

In [ ]:
import os
import json
from datetime import datetime

print("\n💾 SAVING BERT MODEL")
print("=" * 80)

os.makedirs('../trained_models/bert_classifier', exist_ok=True)

# Save fine-tuned model
model.save_pretrained('../trained_models/bert_classifier')
tokenizer.save_pretrained('../trained_models/bert_classifier')

print(f"\n1. Saved fine-tuned BERT:")
print(f"   Location: ../trained_models/bert_classifier/")
print(f"   Includes: model weights, config, tokenizer")

# Save metrics and training metadata
metrics = {
    'timestamp': datetime.now().isoformat(),
    'validation_accuracy': float(val_metrics['accuracy']),
    'validation_precision': float(val_metrics['precision']),
    'validation_recall': float(val_metrics['recall']),
    'validation_f1_score': float(val_metrics['f1']),
    'test_accuracy': float(test_metrics['accuracy']),
    'test_precision': float(test_metrics['precision']),
    'test_recall': float(test_metrics['recall']),
    'test_f1_score': float(test_metrics['f1']),
    'best_model_checkpoint': trainer.state.best_model_checkpoint,
    'best_model_metric': float(trainer.state.best_metric) if trainer.state.best_metric is not None else None,
    'training_time_seconds': float(training_time),
    'training_time_minutes': float(training_time/60),
    'validation_samples': int(len(y_val)),
    'test_samples': int(len(y_test)),
    'training_samples': int(len(y_train)),
    'model_type': 'BERT Transformer (Transfer Learning)',
    'base_model': model_name,
    'num_classes': int(num_classes),
    'max_sequence_length': 128,
    'epochs': 3,
    'train_batch_size': 16,
    'eval_strategy': 'epoch',
    'optimizations': {
        'mixed_precision_fp16': True,
        'dataloader_num_workers': 4,
        'gradient_accumulation_steps': 2,
        'note': 'Full dataset training (1.9M train, 237k val, 237k test)'
    }
}

with open('../trained_models/bert_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n2. Saved metrics:")
print(f"   File: bert_metrics.json")
print(f"   Timestamp: {metrics['timestamp']}")
print(f"   Test F1: {metrics['test_f1_score']:.4f}")
print(f"   Training time: {metrics['training_time_minutes']:.1f} minutes")

print(f"\n3. Dataset information:")
print(f"   Training samples: {metrics['training_samples']:,}")
print(f"   Validation samples: {metrics['validation_samples']:,}")
print(f"   Test samples: {metrics['test_samples']:,}")

print(f"\n4. Optimizations applied:")
print(f"   ✓ FP16 mixed precision (2× speedup)")
print(f"   ✓ 4 dataloader workers (parallel loading)")
print(f"   ✓ Gradient accumulation (2 steps)")
print(f"   ✓ Full dataset training (no sampling)")

print(f"\n✅ BERT training complete!")

In [ ]:
import os
import json
from datetime import datetime

print("\n🔍 FINAL MODEL VERIFICATION")
print("=" * 80)

model_path = '../trained_models/bert_classifier'
metrics_path = '../trained_models/bert_metrics.json'

model_exists = os.path.exists(model_path)
metrics_exists = os.path.exists(metrics_path)

print(f"\n1. Model artifacts:")
print(f"   bert_classifier/ exists: {model_exists}")
print(f"   bert_metrics.json exists: {metrics_exists}")

if model_exists:
    files = os.listdir(model_path)
    print(f"   Files in bert_classifier/: {len(files)} items")
    required_files = ['config.json', 'tokenizer.json', 'tokenizer_config.json']
    for rf in required_files:
        exists = rf in files
        print(f"      {rf}: {'✓' if exists else '❌'}")
    
if metrics_exists:
    metrics = json.load(open(metrics_path))
    print(f"\n2. Training metadata (TRUSTWORTHINESS CHECK):")
    mod_time = datetime.fromisoformat(metrics.get('timestamp', 'unknown'))
    print(f"   Timestamp: {mod_time}")
    
    # Check if training was recent and complete
    if 'training_time_minutes' in metrics:
        train_time = metrics['training_time_minutes']
        print(f"   Training duration: {train_time:.1f} minutes ({train_time/60:.1f} hours)")
        
        # Heuristic: full 3-epoch training on 1.9M samples should take 3+ hours
        if train_time < 60:
            print(f"   ⚠️  WARNING: Unusually short training time (possibly incomplete)")
        else:
            print(f"   ✓ Training time looks complete")
    
    print(f"\n3. Performance metrics:")
    print(f"   Test F1-Score: {metrics.get('test_f1_score', 'N/A'):.4f}")
    print(f"   Test Accuracy: {metrics.get('test_accuracy', 'N/A'):.4f}")
    print(f"   Validation F1: {metrics.get('validation_f1_score', 'N/A'):.4f}")
    
    print(f"\n4. Dataset used (FULL TRAINING):")
    print(f"   Training samples: {metrics.get('training_samples', 'N/A'):,}")
    print(f"   Validation samples: {metrics.get('validation_samples', 'N/A'):,}")
    print(f"   Test samples: {metrics.get('test_samples', 'N/A'):,}")
    
    print(f"\n5. Optimizations applied:")
    opts = metrics.get('optimizations', {})
    print(f"   FP16 precision: {opts.get('mixed_precision_fp16', False)}")
    print(f"   Data workers: {opts.get('dataloader_num_workers', 0)}")
    print(f"   Gradient accumulation: {opts.get('gradient_accumulation_steps', 0)}")
    print(f"   Note: {opts.get('note', 'N/A')}")
    
    # Overall assessment
    print(f"\n6. ✅ MODEL STATUS: READY FOR INFERENCE")
    print(f"   Full dataset training completed with all optimizations.")
else:
    print(f"\n2. ❌ No metrics found - model may not be trustworthy")
    print(f"   The current bert_classifier/ may be from interrupted training.")
    print(f"   Run training cell to generate fresh model with full dataset.")

MODEL EXISTS: True
METRICS EXISTS: True
{'accuracy': 0.9512701674609768, 'precision': 0.9424375861100402, 'recall': 0.9512701674609768, 'f1_score': 0.9464512960794081, 'training_time': 12968.83285164833, 'test_samples': 91484, 'model_type': 'BERT Transformer (Transfer Learning)', 'base_model': 'prajjwal1/bert-tiny', 'num_classes': 15, 'max_sequence_length': 128, 'epochs': 3}
